# Eje 4b — Lectura del panel: IV ATM y descriptivos

Derivados Financieros — QUANt UCEMA

Cómo leer un panel como en la webapp: IV ATM, moneyness, calls vs puts.  
(Usa cadena **US** vía `Codigo.data.market_data`, igual que 04a.)

| Notebook | Contenido |
|----------|-----------|
| **04a** | Spot / quote / cadena **NYSE–US** |
| **04b** (este) | **IV ATM** y descriptivos del panel |
| **04c** | **BYMA**: spot ARS, paneles y opciones |

**Plan del notebook:**
1. Cargar cadena (repaso rápido)
2. IV ATM con fuente
3. Moneyness y filtros
4. r y dividend yield (inputs para pricing)
5. Resumen


In [1]:
import sys
sys.path.append('../../..')

from Codigo.data.market_data import (
    get_spot, get_expirations, get_options_chain,
    get_implied_vol_atm_with_source,
    get_risk_free_rate_with_source, get_dividend_yield,
)

TICKER = 'AAPL'
try:
    spot = get_spot(TICKER)
    exps = get_expirations(TICKER)
    expiry = exps[0]
    chain = get_options_chain(TICKER, expiry)
    print(f'{TICKER} S={spot:.2f} expiry={expiry} rows={len(chain)}')
except Exception as e:
    spot = expiry = chain = None
    print('Sin market data:', type(e).__name__, e)


AAPL S=336.13 expiry=2026-09-21 rows=83


## 2) IV ATM con fuente

La app muestra IV ATM para alimentar escenarios. `get_implied_vol_atm_with_source` devuelve (σ, fuente).


In [2]:
if chain is not None and spot is not None:
    iv = get_implied_vol_atm_with_source(chain, spot, expiry)
    print('IV ATM:', iv)
else:
    print('Saltear: sin cadena')


IV ATM: (0.19605296142578127, 'IV Yahoo ATM')


## 3) Moneyness y filtros

$K/S < 1$ call ITM (aprox.); mirar ambos lados del panel. En la app: filtros de strike + smile de IV.


In [3]:
if chain is not None and spot is not None:
    df = chain.copy()
    df['moneyness'] = df['strike'] / spot
    # band near ATM
    near = df[(df['moneyness'] > 0.9) & (df['moneyness'] < 1.1)]
    print(f'Filas near-ATM: {len(near)} / {len(df)}')
    cols = [c for c in ['strike', 'moneyness', 'call_last', 'put_last', 'call_iv', 'put_iv'] if c in near.columns]
    display(near[cols].head(12) if cols else near.head())
else:
    print('Saltear')


Filas near-ATM: 47 / 83


,strike,moneyness
15,305.0,0.907387
16,307.5,0.914825
17,310.0,0.922262
18,312.5,0.929700
19,315.0,0.937137
20,317.5,0.944575
21,320.0,0.952013
22,322.5,0.959450
23,325.0,0.966888
24,327.5,0.974325


## 4) r y dividend yield

Inputs que usa *Market Data Pricing* para escenarios. No hace falta BS completo todavía (Eje 6).


In [4]:
try:
    r, r_src = get_risk_free_rate_with_source()
    print(f'r = {r:.4%}  ({r_src})')
except Exception as e:
    print('r no disponible:', e)

try:
    if spot is not None:
        div = get_dividend_yield(TICKER)
        print(f'div yield ≈ {div:.4%}')
except Exception as e:
    print('div no disponible:', e)


r = 3.9780%  (^IRX)

429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=AAPL&crumb=Edge%3A+Too+Many+Requests


div yield ≈ 0.0000%


## Resumen

- IV ATM + moneyness = lectura profesional del panel.
- r y div alimentan valuación (app *Market Data Pricing* y Eje 3 con precios reales).
- **Paralelo AR:** **04c** — mismos conceptos sobre paneles BYMA/data912.
- **Siguiente en el curso:** Eje 5 binomial · Eje 6 BS · Eje 7 griegas · Eje 8 volatilidad.
